# 02 - Document Ingestion Pipeline

This notebook demonstrates the LlamaIndex ingestion pipeline that:
1. Loads scanned PDF agreements
2. Enriches metadata (language, document type)
3. Chunks documents with `SentenceSplitter`
4. Embeds and upserts to Pinecone

**Prerequisites:** Run `01_generate_data.ipynb` first to create the scanned PDFs.

In [ ]:
import sys
sys.path.insert(0, '../src')
from pathlib import Path

from contract_lens.config import get_settings
from contract_lens.observability import init_observability

In [ ]:
settings = get_settings()
init_observability(settings)
print(f'LLM deployment:       {settings.azure_openai_deployment}')
print(f'Embedding deployment: {settings.azure_openai_embedding_deployment}')
print(f'Pinecone index:       {settings.pinecone_index_name}')
print(f'LangFuse enabled:     {settings.langfuse_enabled}')

## Step 1: Load Scanned PDFs

LlamaIndex's `SimpleDirectoryReader` reads all PDFs from `data/scans/`.
Each page becomes a separate `Document` object with metadata.

In [ ]:
from llama_index.core import SimpleDirectoryReader

data_path = Path('../data/scans')
reader = SimpleDirectoryReader(input_dir=str(data_path), recursive=False)
documents = reader.load_data()

print(f'Loaded {len(documents)} document pages from {data_path}')
print(f'\nFirst document metadata: {documents[0].metadata}')
print(f'First document text (first 300 chars):\n{documents[0].text[:300]}')

## Step 2: Enrich Metadata

We detect **language** (EN/PL) and **document type** (agreement/annex) from filenames.
This metadata enables filtered retrieval later.

In [ ]:
from contract_lens.ingestion.pipeline import _detect_language, _detect_document_type

for doc in documents:
    filename = doc.metadata.get('file_name', '')
    doc.metadata['language'] = _detect_language(filename)
    doc.metadata['document_type'] = _detect_document_type(filename)

# Show metadata summary
seen = set()
for doc in documents:
    fn = doc.metadata.get('file_name', '')
    if fn not in seen:
        seen.add(fn)
        print(f"{fn:45s} lang={doc.metadata['language']}  type={doc.metadata['document_type']}")

## Step 3: Chunk Documents

We use `SentenceSplitter` with:
- **chunk_size=1024** tokens — large enough to preserve full contract clauses
- **chunk_overlap=128** tokens — ensures context continuity across chunk boundaries

In [ ]:
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(chunk_size=1024, chunk_overlap=128)
nodes = splitter.get_nodes_from_documents(documents)

print(f'{len(documents)} pages -> {len(nodes)} chunks')
print(f'\nSample chunk (node 0):')
print(f'  Text length: {len(nodes[0].text)} chars')
print(f'  Metadata:    {nodes[0].metadata}')
print(f'  Text:        {nodes[0].text[:200]}...')

## Step 4: Embed and Upsert to Pinecone

Azure OpenAI `text-embedding-3-large` creates vector embeddings.
These are upserted to Pinecone along with the text and metadata.

In [ ]:
from llama_index.core import StorageContext, VectorStoreIndex
from contract_lens.ingestion.pipeline import build_embedding_model, build_pinecone_vector_store

embed_model = build_embedding_model(settings)
vector_store = build_pinecone_vector_store(settings)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex(
    nodes,
    embed_model=embed_model,
    storage_context=storage_context,
    show_progress=True,
)

print(f'\nSuccessfully embedded and upserted {len(nodes)} chunks to Pinecone.')

## Step 5: Quick Verification

Run a quick query to verify the index is working.

In [ ]:
from contract_lens.retrieval.query_engine import query_contracts

answer = query_contracts(settings, question='What types of agreements are in the knowledge base?')
print(answer)

## One-Shot Alternative

All the steps above are wrapped in a single function for production use:

In [ ]:
# from contract_lens.ingestion.pipeline import run_ingestion
# num_nodes = run_ingestion(settings, data_dir='../data/scans')
# print(f'Ingested {num_nodes} chunks')

## Summary

| Step | Component | Result |
|------|-----------|--------|
| Load | `SimpleDirectoryReader` | PDF pages as Documents |
| Metadata | `_detect_language` / `_detect_document_type` | Language + type tags |
| Chunk | `SentenceSplitter(1024/128)` | Overlapping text nodes |
| Embed + Store | `AzureOpenAIEmbedding` + `PineconeVectorStore` | Vectors in Pinecone |

**Next:** Run `03_agent_demo.ipynb` to interact with the LangGraph agent.